# Audio files in your browser: libsndfile + python-soundfile

[libsndfile](https://libsndfile.github.io/libsndfile/) is the standard C library for reading and writing sound files, and [python-soundfile](https://python-soundfile.readthedocs.io) is its Python wrapper that returns numpy arrays. In this notebook we synthesize a little melody, write it as a WAV file into the browser's file system, read it back, look at it, and listen to it.

You can also upload your own WAV/AIFF file in the file browser on the left and read it with `sf.read(...)`. (FLAC/Ogg/MP3 are not supported yet in this build.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio

print(sf.__libsndfile_version__)
sf.available_formats()

## Synthesize a melody

In [ ]:
rate = 22050


def note(freq, duration=0.3):
    t = np.arange(int(rate * duration)) / rate
    envelope = np.exp(-4 * t)  # a plucked, decaying sound
    tone = np.sin(2 * np.pi * freq * t) + 0.3 * np.sin(2 * np.pi * 2 * freq * t)
    return envelope * tone


# C major arpeggio up and down
freqs = [261.63, 329.63, 392.00, 523.25, 392.00, 329.63, 261.63]
melody = np.concatenate([note(f) for f in freqs])
melody /= np.abs(melody).max()

# stereo: the right channel is slightly delayed
stereo = np.stack([melody, np.roll(melody, 2000)], axis=1)
stereo.shape

## Write it to a WAV file and read it back

In [ ]:
sf.write("melody.wav", stereo, rate, subtype="PCM_16")
print(sf.info("melody.wav"))

data, file_rate = sf.read("melody.wav")
data.shape, file_rate

`melody.wav` now shows up in the file browser on the left; you can download it from there.

## Listen

In [ ]:
Audio(data.T, rate=file_rate)

## Look at it: waveform and spectrogram

In [ ]:
left = data[:, 0]
t = np.arange(len(left)) / file_rate

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(t, left, linewidth=0.5)
ax1.set_ylabel("amplitude")
ax1.set_title("waveform (left channel)")
ax2.specgram(left, NFFT=1024, Fs=file_rate, noverlap=768, cmap="magma")
ax2.set_ylim(0, 2000)
ax2.set_xlabel("time (s)")
ax2.set_ylabel("frequency (Hz)")
ax2.set_title("spectrogram")
fig.tight_layout()

## Process audio block by block

`sf.blocks` streams a file in chunks, which is handy for long recordings. Here we compute the loudness (RMS) of every 50 ms block.

In [ ]:
block = int(0.05 * file_rate)
rms = [np.sqrt(np.mean(b**2)) for b in sf.blocks("melody.wav", blocksize=block)]

plt.figure(figsize=(10, 3))
plt.bar(np.arange(len(rms)) * 0.05, rms, width=0.045)
plt.xlabel("time (s)")
plt.ylabel("RMS")
plt.title("loudness per 50 ms block")

## Read a file that ships with this deployment

`sample.wav` sits next to this notebook in the file browser. Upload your own WAV or AIFF file the same way to analyze your own audio.

In [ ]:
print(sf.info("sample.wav"))
sample, sample_rate = sf.read("sample.wav")
Audio(sample.T, rate=sample_rate)

Other formats work too, for example `sf.write("melody.aiff", stereo, rate)` or `sf.write("melody.au", stereo, rate)`.